# 🔍 NCO Job Title — Embedding Model Benchmark
**Benchmarking 6 sentence-transformer models on 35,239 multilingual NCO queries**

| Model | Dim | Speed | Multilingual |
|---|---|---|---|
| all-MiniLM-L6-v2 | 384 | ⭐⭐⭐⭐⭐ | No |
| BAAI/bge-small-en-v1.5 | 384 | ⭐⭐⭐⭐ | No |
| thenlper/gte-base | 768 | ⭐⭐⭐ | Limited |
| intfloat/multilingual-e5-base | 768 | ⭐⭐⭐ | Yes |
| BAAI/bge-large-en-v1.5 | 1024 | ⭐⭐ | No |
| hkunlp/instructor-xl | 768 | ⭐ | Partial |

**Metrics collected:** Top-1/3/5 Accuracy · MRR · Encode Speed · Query Latency · Peak GPU/RAM  
**Breakdowns:** by Language · Query Type · Difficulty  
**Expected runtime on T4 GPU:** ~45–60 min for all 6 models


In [ ]:
# Cell 1 — Install / verify dependencies
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import sentence_transformers
    print(f"sentence-transformers {sentence_transformers.__version__} ✓")
except ImportError:
    print("Installing sentence-transformers...")
    pip_install("sentence-transformers")

try:
    import psutil
    print(f"psutil {psutil.__version__} ✓")
except ImportError:
    pip_install("psutil")

# instructor-xl needs this extra package
try:
    from InstructorEmbedding import INSTRUCTOR
    print("InstructorEmbedding ✓")
except ImportError:
    print("Installing InstructorEmbedding (needed for instructor-xl)...")
    pip_install("InstructorEmbedding")

print("\nAll dependencies ready.")


In [ ]:
# Cell 2 — Imports & GPU check
import os, gc, time, warnings
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

print(f"PyTorch version   : {torch.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"RAM available     : {psutil.virtual_memory().available / 1e9:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nRunning on        : {DEVICE.upper()}")


In [ ]:
# Cell 3 — Configuration (edit paths here if needed)

# ── Dataset path ──────────────────────────────────────────────────────────────
# Kaggle: upload benchmark_dataset_multilingual.csv as dataset "nco-benchmark"
DATASET_PATH = "/kaggle/input/nco-benchmark/benchmark_dataset_multilingual.csv"
if not os.path.exists(DATASET_PATH):
    # Local fallback
    DATASET_PATH = str(Path(__file__).parent.parent / "data" / "raw" / "benchmark_dataset_multilingual.csv") \
        if "__file__" in dir() else "data/raw/benchmark_dataset_multilingual.csv"

OUTPUT_DIR = Path("/kaggle/working") if os.path.exists("/kaggle") else Path("data/benchmark_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K_VALUES = [1, 3, 5]

# ── Model configs (batch_size tuned for T4 16GB VRAM) ────────────────────────
MODELS = [
    {
        "name": "all-MiniLM-L6-v2",
        "hf_id": "all-MiniLM-L6-v2",
        "dim": 384,
        "batch_size": 512,
        "notes": "Fastest, lowest memory",
    },
    {
        "name": "BAAI/bge-small-en-v1.5",
        "hf_id": "BAAI/bge-small-en-v1.5",
        "dim": 384,
        "batch_size": 512,
        "notes": "Fast, strong accuracy",
    },
    {
        "name": "thenlper/gte-base",
        "hf_id": "thenlper/gte-base",
        "dim": 768,
        "batch_size": 256,
        "notes": "Medium size, good accuracy",
    },
    {
        "name": "intfloat/multilingual-e5-base",
        "hf_id": "intfloat/multilingual-e5-base",
        "dim": 768,
        "batch_size": 256,
        "notes": "Best multilingual coverage",
        "query_prefix": "query: ",       # e5 requires these prefixes
        "corpus_prefix": "passage: ",
    },
    {
        "name": "BAAI/bge-large-en-v1.5",
        "hf_id": "BAAI/bge-large-en-v1.5",
        "dim": 1024,
        "batch_size": 128,
        "notes": "High accuracy, larger model",
    },
    {
        "name": "hkunlp/instructor-xl",
        "hf_id": "hkunlp/instructor-xl",
        "dim": 768,
        "batch_size": 64,
        "notes": "Highest accuracy, instruction-tuned",
        "instruction_query": "Represent the job search query for finding the NCO job title: ",
        "instruction_corpus": "Represent the NCO job title for semantic search: ",
    },
]

print(f"Dataset : {DATASET_PATH}")
print(f"Output  : {OUTPUT_DIR}")
print(f"Models  : {len(MODELS)}")
print(f"Top-K   : {TOP_K_VALUES}")


In [ ]:
# Cell 4 — Load dataset & build corpus
print("=" * 60)
print("LOADING DATASET")
print("=" * 60)

df = pd.read_csv(DATASET_PATH, encoding="utf-8-sig")
df.columns = df.columns.str.strip()

print(f"Total rows    : {len(df):,}")
print(f"Languages     : {sorted(df['language'].unique())}")
print(f"Query types   : {sorted(df['query_type'].unique())}")
print(f"Difficulties  : {sorted(df['difficulty'].unique())}")

# ── Corpus: 3,447 unique NCO job titles ──────────────────────────────────────
corpus_df    = df[["expected_nco_code", "expected_title"]].drop_duplicates()
corpus_df    = corpus_df.sort_values("expected_nco_code").reset_index(drop=True)
CORPUS_TEXTS = corpus_df["expected_title"].tolist()
CORPUS_CODES = corpus_df["expected_nco_code"].tolist()
CODE_TO_IDX  = {code: idx for idx, code in enumerate(CORPUS_CODES)}

# ── Queries ───────────────────────────────────────────────────────────────────
QUERIES     = df["query"].tolist()
QUERY_CODES = df["expected_nco_code"].tolist()

print(f"\nCorpus size   : {len(CORPUS_TEXTS):,} unique NCO titles")
print(f"Query count   : {len(QUERIES):,}")


In [ ]:
# Cell 5 — Helper functions

def get_gpu_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e6
    return 0.0

def get_ram_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1e6

def flush_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    gc.collect()

def encode_texts(model, texts, batch_size, prefix=None, instruction=None):
    """
    Encode texts with optional e5-prefix or instructor-instruction.
    Returns L2-normalised numpy float32 array.
    """
    if instruction is not None:
        inputs = [[instruction, t] for t in texts]
    elif prefix is not None:
        inputs = [prefix + t for t in texts]
    else:
        inputs = texts

    return model.encode(
        inputs,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=DEVICE,
    )

def top_k_search(query_embs, corpus_embs, chunk_size=2000):
    """
    GPU-accelerated chunked cosine similarity search.
    L2-normalised vectors → cosine sim = dot product.
    Returns top-K indices and scores for every query.
    """
    max_k = max(TOP_K_VALUES)
    n = len(query_embs)
    top_indices = np.zeros((n, max_k), dtype=np.int32)
    top_scores  = np.zeros((n, max_k), dtype=np.float32)

    corpus_t = torch.tensor(corpus_embs, dtype=torch.float32)
    if DEVICE == "cuda":
        corpus_t = corpus_t.cuda()

    for start in range(0, n, chunk_size):
        end   = min(start + chunk_size, n)
        chunk = torch.tensor(query_embs[start:end], dtype=torch.float32)
        if DEVICE == "cuda":
            chunk = chunk.cuda()
        sims  = torch.mm(chunk, corpus_t.T)
        topk  = torch.topk(sims, k=max_k, dim=1)
        top_indices[start:end] = topk.indices.cpu().numpy()
        top_scores[start:end]  = topk.values.cpu().numpy()
        del chunk, sims, topk

    del corpus_t
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return top_indices, top_scores

def compute_metrics(top_indices, query_codes, df):
    """
    Compute Top-K accuracy, MRR, and per-slice breakdowns.
    Returns (summary_dict, per_query_list).
    """
    per_query = []
    rr_list   = []
    slice_keys = ["language", "query_type", "difficulty"]
    slice_hits = {sk: defaultdict(lambda: defaultdict(list)) for sk in slice_keys}
    slice_rr   = {sk: defaultdict(list) for sk in slice_keys}

    for i, (correct_code, pred_indices) in enumerate(zip(query_codes, top_indices)):
        correct_idx = CODE_TO_IDX.get(correct_code, -1)
        rank = -1
        for r, idx in enumerate(pred_indices):
            if idx == correct_idx:
                rank = r + 1
                break

        rr = 1.0 / rank if rank > 0 else 0.0
        rr_list.append(rr)
        row = df.iloc[i]
        rec = {
            "query_id": row["query_id"], "language": row["language"],
            "query_type": row["query_type"], "difficulty": row["difficulty"],
            "correct_code": correct_code, "rank": rank, "reciprocal_rank": rr,
        }
        for k in TOP_K_VALUES:
            rec[f"hit@{k}"] = 1 if (0 < rank <= k) else 0
        per_query.append(rec)

        for sk in slice_keys:
            sv = row[sk]
            for k in TOP_K_VALUES:
                slice_hits[sk][sv][k].append(rec[f"hit@{k}"])
            slice_rr[sk][sv].append(rr)

    summary = {f"top{k}_accuracy": round(np.mean([r[f"hit@{k}"] for r in per_query]) * 100, 2)
               for k in TOP_K_VALUES}
    summary["mrr"] = round(np.mean(rr_list), 4)

    for sk in slice_keys:
        for sv, kdict in slice_hits[sk].items():
            for k in TOP_K_VALUES:
                summary[f"{sk}_{sv}_top{k}"] = round(np.mean(kdict[k]) * 100, 2)
        for sv, rrs in slice_rr[sk].items():
            summary[f"{sk}_{sv}_mrr"] = round(np.mean(rrs), 4)

    return summary, per_query

print("Helper functions defined ✓")


In [ ]:
# Cell 6 — Main benchmark loop (runs all 6 models)
print("=" * 60)
print("STARTING BENCHMARK")
print("=" * 60)

all_summaries = []
all_per_query = []

for m_cfg in MODELS:
    model_name = m_cfg["name"]
    hf_id      = m_cfg["hf_id"]
    batch_size = m_cfg["batch_size"]

    print(f"\n{'─'*60}")
    print(f"▶ Model : {model_name}")
    print(f"  Notes : {m_cfg.get('notes','')}")
    print(f"{'─'*60}")

    flush_gpu()
    ram_before = get_ram_mb()
    gpu_before = get_gpu_memory_mb()

    # Load model
    t0 = time.time()
    model = SentenceTransformer(hf_id, device=DEVICE)
    load_time = time.time() - t0
    print(f"  ✓ Loaded in {load_time:.1f}s")

    # Encode corpus
    print(f"  Encoding {len(CORPUS_TEXTS):,} corpus titles...")
    t1 = time.time()
    corpus_embs = encode_texts(model, CORPUS_TEXTS, batch_size,
                               prefix=m_cfg.get("corpus_prefix"),
                               instruction=m_cfg.get("instruction_corpus"))
    corpus_time  = time.time() - t1
    corpus_speed = len(CORPUS_TEXTS) / corpus_time
    print(f"  ✓ Corpus done  {corpus_time:.1f}s  ({corpus_speed:.0f} docs/sec)")

    # Encode queries
    print(f"  Encoding {len(QUERIES):,} queries...")
    t2 = time.time()
    query_embs = encode_texts(model, QUERIES, batch_size,
                              prefix=m_cfg.get("query_prefix"),
                              instruction=m_cfg.get("instruction_query"))
    query_time    = time.time() - t2
    query_speed   = len(QUERIES) / query_time
    latency_ms    = (query_time / len(QUERIES)) * 1000
    print(f"  ✓ Queries done {query_time:.1f}s  ({query_speed:.0f} q/sec | {latency_ms:.3f} ms/query)")

    peak_ram_mb = get_ram_mb() - ram_before
    peak_gpu_mb = get_gpu_memory_mb() - gpu_before

    # Similarity search
    print(f"  Running top-{max(TOP_K_VALUES)} search...")
    t3 = time.time()
    top_indices, top_scores = top_k_search(query_embs, corpus_embs)
    search_time = time.time() - t3
    print(f"  ✓ Search done  {search_time:.1f}s")

    # Score
    print("  Computing metrics...")
    metrics, per_query_rows = compute_metrics(top_indices, QUERY_CODES, df)

    total_time = load_time + corpus_time + query_time + search_time
    summary = {
        "model": model_name, "embedding_dim": corpus_embs.shape[1],
        "device": DEVICE,
        "total_time_sec": round(total_time, 1),
        "load_time_sec": round(load_time, 1),
        "corpus_encode_sec": round(corpus_time, 1),
        "corpus_speed_docs_sec": round(corpus_speed, 0),
        "query_encode_sec": round(query_time, 1),
        "query_speed_q_sec": round(query_speed, 0),
        "avg_latency_ms": round(latency_ms, 3),
        "search_time_sec": round(search_time, 1),
        "peak_ram_delta_mb": round(peak_ram_mb, 1),
        "peak_gpu_delta_mb": round(peak_gpu_mb, 1),
        **metrics,
    }
    all_summaries.append(summary)
    for row in per_query_rows:
        row["model"] = model_name
    all_per_query.extend(per_query_rows)

    # Print results
    print(f"\n  ┌─ RESULTS ─────────────────────────────────────────")
    print(f"  │  Top-1 Accuracy  : {metrics['top1_accuracy']}%")
    print(f"  │  Top-3 Accuracy  : {metrics['top3_accuracy']}%")
    print(f"  │  Top-5 Accuracy  : {metrics['top5_accuracy']}%")
    print(f"  │  MRR             : {metrics['mrr']}")
    print(f"  │  Total Time      : {total_time:.1f}s")
    print(f"  │  Peak RAM delta  : {peak_ram_mb:.0f} MB")
    if DEVICE == "cuda":
        print(f"  │  Peak GPU delta  : {peak_gpu_mb:.0f} MB")
    print(f"  └───────────────────────────────────────────────────")

    # Checkpoint — safe even if session dies
    chk = OUTPUT_DIR / f"checkpoint_{model_name.replace('/', '_')}.csv"
    pd.DataFrame([summary]).to_csv(chk, index=False)
    print(f"  ✓ Checkpoint → {chk.name}")

    del model, corpus_embs, query_embs, top_indices, top_scores
    flush_gpu()

print("\n✅ All models done.")


In [ ]:
# Cell 7 — Save results to CSV
print("=" * 60)
print("SAVING RESULTS")
print("=" * 60)

summary_df  = pd.DataFrame(all_summaries)
detailed_df = pd.DataFrame(all_per_query)

summary_path  = OUTPUT_DIR / "benchmark_results_summary.csv"
detailed_path = OUTPUT_DIR / "benchmark_results_detailed.csv"

summary_df.to_csv(summary_path, index=False)
detailed_df.to_csv(detailed_path, index=False)

print(f"✓ Summary  ({len(summary_df)} rows)  → {summary_path}")
print(f"✓ Detailed ({len(detailed_df):,} rows) → {detailed_path}")


In [ ]:
# Cell 8 — Generate Markdown report
def generate_report(sdf):
    now    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ranked = sdf.sort_values("top1_accuracy", ascending=False)
    lines  = [
        "# NCO Embedding Model Benchmark Report",
        f"*Generated: {now} | Device: {DEVICE.upper()} | Queries: {len(df):,} | Corpus: {len(CORPUS_TEXTS):,}*\n",
    ]

    # ── Overall table ──────────────────────────────────────────────────────────
    lines.append("## 📊 Overall Rankings\n")
    cols   = ["model","top1_accuracy","top3_accuracy","top5_accuracy","mrr",
              "avg_latency_ms","corpus_speed_docs_sec","peak_ram_delta_mb","total_time_sec"]
    labels = ["Model","Top-1 %","Top-3 %","Top-5 %","MRR",
              "Latency (ms)","Speed (doc/s)","RAM δ (MB)","Time (s)"]
    lines.append("| " + " | ".join(labels) + " |")
    lines.append("| " + " | ".join(["---"]*len(labels)) + " |")
    for _, r in ranked.iterrows():
        lines.append("| " + " | ".join(str(r.get(c,"N/A")) for c in cols) + " |")
    lines.append("")

    # ── Bests ─────────────────────────────────────────────────────────────────
    lines.append("## 🏆 Best Model per Criterion\n")
    bests = {
        "Highest Top-1 Accuracy": sdf.loc[sdf["top1_accuracy"].idxmax(),"model"],
        "Highest MRR":            sdf.loc[sdf["mrr"].idxmax(),"model"],
        "Fastest (q/sec)":        sdf.loc[sdf["query_speed_q_sec"].idxmax(),"model"],
        "Lowest Latency":         sdf.loc[sdf["avg_latency_ms"].idxmin(),"model"],
        "Lowest RAM":             sdf.loc[sdf["peak_ram_delta_mb"].idxmin(),"model"],
    }
    for k, v in bests.items():
        lines.append(f"- **{k}**: `{v}`")
    lines.append("")

    # ── By Language ───────────────────────────────────────────────────────────
    lines.append("## 🌍 Top-1 Accuracy by Language\n")
    langs = sorted({c.split("_")[1] for c in sdf.columns
                    if c.startswith("language_") and c.endswith("_top1")})
    if langs:
        lines.append("| Model | " + " | ".join(langs) + " |")
        lines.append("| --- | " + " | ".join(["---"]*len(langs)) + " |")
        for _, r in ranked.iterrows():
            vals = [f"{r.get(f'language_{l}_top1','N/A')}%" for l in langs]
            lines.append(f"| {r['model']} | " + " | ".join(vals) + " |")
        lines.append("")

    # ── By Query Type ─────────────────────────────────────────────────────────
    lines.append("## 🔍 Top-1 Accuracy by Query Type\n")
    qtypes = sorted({"_".join(c.split("_")[1:-1]) for c in sdf.columns
                     if c.startswith("query_type_") and c.endswith("_top1")})
    if qtypes:
        lines.append("| Model | " + " | ".join(qtypes) + " |")
        lines.append("| --- | " + " | ".join(["---"]*len(qtypes)) + " |")
        for _, r in ranked.iterrows():
            vals = [f"{r.get(f'query_type_{q}_top1','N/A')}%" for q in qtypes]
            lines.append(f"| {r['model']} | " + " | ".join(vals) + " |")
        lines.append("")

    # ── By Difficulty ─────────────────────────────────────────────────────────
    lines.append("## 🎯 Top-1 Accuracy by Difficulty\n")
    lines.append("| Model | Easy | Medium | Hard |")
    lines.append("| --- | --- | --- | --- |")
    for _, r in ranked.iterrows():
        vals = [f"{r.get(f'difficulty_{d}_top1','N/A')}%" for d in ["easy","medium","hard"]]
        lines.append(f"| {r['model']} | " + " | ".join(vals) + " |")
    lines.append("")

    # ── Recommendation ────────────────────────────────────────────────────────
    lines.append("## ✅ Recommendation\n")
    best_acc  = sdf.loc[sdf["top1_accuracy"].idxmax(),"model"]
    best_bal  = sdf.loc[(sdf["top1_accuracy"]/sdf["avg_latency_ms"]).idxmax(),"model"]
    ml_col    = "language_Hindi_top1" if "language_Hindi_top1" in sdf.columns else None
    best_ml   = sdf.loc[sdf[ml_col].idxmax(),"model"] if ml_col else "N/A"
    lines += [
        "| Use Case | Recommended Model |",
        "| --- | --- |",
        f"| Best accuracy | `{best_acc}` |",
        f"| Best accuracy/speed balance | `{best_bal}` |",
        f"| Best multilingual | `{best_ml}` |",
        "",
        f"> **Next step:** Update `MODEL_NAME = \"{best_acc}\"` in `scripts/legacy_pipeline/02_generateEmbedding.py`",
    ]
    return "\n".join(lines)

report_md   = generate_report(summary_df)
report_path = OUTPUT_DIR / "benchmark_report.md"
report_path.write_text(report_md, encoding="utf-8")
print(f"✓ Report → {report_path}")
print("\nPreview (first 40 lines):")
print("\n".join(report_md.splitlines()[:40]))


In [ ]:
# Cell 9 — Final ranked summary table
print("=" * 60)
print("BENCHMARK COMPLETE — FINAL SUMMARY")
print("=" * 60)

display_cols = ["model","top1_accuracy","top3_accuracy","top5_accuracy",
                "mrr","avg_latency_ms","total_time_sec"]

ranked = summary_df.sort_values("top1_accuracy", ascending=False)
print("\n" + ranked[display_cols].to_string(index=False))

print(f"\nOutput files saved to: {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.glob("benchmark_*")):
    size_kb = f.stat().st_size // 1024
    print(f"  {f.name:<45} {size_kb:>6} KB")
